# Week 39, Day 1: Review: classes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w039d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Counter:
    made = 0

    def __init__(self, start=0):
        self.count = start
        Counter.made += 1

    def tick(self):
        self.count += 1
        return self

a = Counter()
b = Counter(10)
a.tick().tick()
b.made = 99
print(a.count, b.count, Counter.made, a.made, b.made)

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius            # goes through the setter

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(f"below absolute zero: {value}")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

t = Temperature(20)
t.celsius = 100
print(t.celsius, t.fahrenheit)
try:
    Temperature(-300)
except ValueError as err:
    print(err)

In [ ]:
from functools import total_ordering

@total_ordering
class Money:
    def __init__(self, pence):
        self.pence = pence

    def __repr__(self):
        return f"Money({self.pence})"

    def __str__(self):
        return f"£{self.pence // 100}.{self.pence % 100:02d}"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.pence == other.pence

    def __hash__(self):
        return hash(self.pence)

    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.pence < other.pence

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.pence + other.pence)

    def __mul__(self, n):
        return Money(self.pence * n)

    __rmul__ = __mul__

    def __bool__(self):
        return self.pence != 0

prices = [Money(250), Money(99), Money(250)]
print(prices, sorted(prices), len(set(prices)))
print(sum(prices, Money(0)), 3 * Money(120), Money(5) >= Money(5), bool(Money(0)))
print(Money(5) == 5)

**Predict before you run:** what will this print?



In [ ]:
class Playlist:
    def __init__(self, *songs):
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, i):
        return self._songs[i]

p = Playlist("Zombie", "Water No Get Enemy", "Lady")
print(len(p), p[0], p[-1], p[1:])
print("Lady" in p, [s.upper() for s in p][:1], list(reversed(p)))

**Predict before you run:** what will this print?



In [ ]:
class A:
    def hello(self):
        return "A"

class B(A):
    def hello(self):
        return "B" + super().hello()

class C(A):
    def hello(self):
        return "C" + super().hello()

class D(B, C):
    def hello(self):
        return "D" + super().hello()

print(D().hello(), [k.__name__ for k in D.__mro__])

In [ ]:
class Engine:
    def start(self):
        return "vroom"

class ElectricEngine:
    def start(self):
        return "hum"

class Car:                        # has an engine, rather than being one
    def __init__(self, engine):
        self.engine = engine

    def start(self):
        return f"car: {self.engine.start()}"

print(Car(Engine()).start(), Car(ElectricEngine()).start())

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

print(Point(1, 2) == Point(1, 2))
print({Point(1, 2)})

> **Key idea.** A class bundles data and methods; `self` is the instance, class attributes are shared, and properties guard or compute values. Special methods connect objects to Python's syntax: `__repr__`/`__str__`, `__eq__` with a matching `__hash__`, `__lt__` with `total_ordering`, arithmetic with `__add__`/`__radd__`, and `NotImplemented` for unknown types. `__len__`, `__getitem__`, `__contains__` and `__iter__` make containers. Inherit for "is a", call `super()` (the next class in the MRO), and compose for "has a".

### Exercise: A fraction class

Write `Fraction(top, bottom)`, kept in lowest terms with a positive bottom (use `math.gcd`; a bottom of 0 raises `ZeroDivisionError`). Give it `__repr__` (`Fraction(1, 2)`), `__str__` (`1/2`, or just `3` when the bottom is 1), `__eq__` and `__hash__`, `__lt__`, and `__add__` and `__mul__` with other fractions **and** with ints (an int `n` is `n/1`), including `1 + Fraction(1, 2)`. Return `NotImplemented` for anything else.

In [ ]:
from math import gcd

class Fraction:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"lowest terms\"\n    f = Fraction(6, -8)\n    assert (f.top, f.bottom) == (-3, 4) and repr(f) == \"Fraction(-3, 4)\" and str(f) == \"-3/4\" and str(Fraction(4, 2)) == \"2\"\n    try:\n        Fraction(1, 0)\n        assert False\n    except ZeroDivisionError:\n        pass\ndef test_compare():\n    \"equality, hashing and order\"\n    assert Fraction(1, 2) == Fraction(2, 4) and len({Fraction(1, 2), Fraction(3, 6), Fraction(1, 3)}) == 2\n    assert Fraction(1, 3) < Fraction(1, 2) and not Fraction(1, 2) < Fraction(1, 2) and (Fraction(1, 2) == \"1/2\") is False\n    assert sorted([Fraction(3, 4), Fraction(-1, 2), Fraction(1, 3)]) == [Fraction(-1, 2), Fraction(1, 3), Fraction(3, 4)]\ndef test_maths():\n    \"arithmetic\"\n    assert Fraction(1, 2) + Fraction(1, 3) == Fraction(5, 6) and Fraction(1, 2) + 1 == Fraction(3, 2) and 1 + Fraction(1, 2) == Fraction(3, 2)\n    assert Fraction(2, 3) * Fraction(3, 4) == Fraction(1, 2) and 3 * Fraction(1, 6) == Fraction(1, 2)\n    assert sum([Fraction(1, 4)] * 4, Fraction(0, 1)) == Fraction(1, 1)\n    try:\n        Fraction(1, 2) + \"x\"\n        assert False\n    except TypeError:\n        pass")

### Exercise: A grid you can index

Write `Grid(rows, cols, fill=".")`, a container of cells. `g[r, c]` reads a cell and `g[r, c] = value` sets it (the key is a tuple); a position outside the grid raises `IndexError`. `len(g)` is the number of cells, `(r, c) in g` is true for positions inside the grid, iterating gives the rows as strings (cells joined with no spaces), and `str(g)` is those rows joined by newlines.

In [ ]:
class Grid:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_grid():\n    \"a grid\"\n    g = Grid(2, 3)\n    g[0, 1] = \"#\"\n    g[1, 2] = \"@\"\n    assert g[0, 1] == \"#\" and g[1, 0] == \".\" and len(g) == 6\n    assert list(g) == [\".#.\", \"..@\"] and str(g) == \".#.\\n..@\"\n    assert (1, 2) in g and (2, 0) not in g and (0, -1) not in g\n    for pos in [(2, 0), (0, 3), (-1, 0)]:\n        try:\n            g[pos]\n            assert False, f\"{pos} is outside\"\n        except IndexError:\n            pass\n    try:\n        g[5, 5] = \"x\"\n        assert False\n    except IndexError:\n        pass\n    assert str(Grid(1, 2, \"o\")) == \"oo\"")

### Exercise: Accounts and a subclass

Write `Account(owner, balance=0)` with `deposit(amount)` and `withdraw(amount)` (both refuse amounts of 0 or less with `ValueError`; `withdraw` refuses more than the balance with `ValueError("insufficient funds")`), a read-only property `balance`, and `__repr__` like `Account('Ada', 50)`. Then `Overdraft(owner, balance=0, limit=100)`, a subclass that may go down to `-limit`: call `super().__init__`, and override **only** what it needs. Its repr should start with `Overdraft(` without you writing a new `__repr__`.

In [ ]:
class Account:
    ___

class Overdraft(Account):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_account():\n    \"accounts\"\n    a = Account(\"Ada\", 50)\n    a.deposit(25)\n    a.withdraw(70)\n    assert a.balance == 5 and repr(a) == \"Account('Ada', 5)\"\n    for call in [lambda: a.withdraw(6), lambda: a.deposit(0), lambda: a.withdraw(-1)]:\n        try:\n            call()\n            assert False\n        except ValueError:\n            pass\n    try:\n        a.balance = 1000\n        assert False, \"balance should be read-only\"\n    except AttributeError:\n        pass\ndef test_overdraft():\n    \"overdrafts\"\n    o = Overdraft(\"Bo\", 20, limit=50)\n    o.withdraw(60)\n    assert o.balance == -40 and repr(o) == \"Overdraft('Bo', -40)\" and isinstance(o, Account)\n    try:\n        o.withdraw(11)\n        assert False\n    except ValueError as err:\n        assert str(err) == \"insufficient funds\"\n    assert \"__repr__\" not in Overdraft.__dict__ and \"deposit\" not in Overdraft.__dict__")

### Exercise: Composition, not inheritance

Write `Stack`, which **holds** a list rather than inheriting from one, so it offers only `push(item)`, `pop()` (raising `IndexError("pop from an empty stack")` when empty), `peek()` (the top item, same error), `__len__`, `__bool__` and `__repr__` like `Stack([1, 2, 3])` (bottom first).

In [ ]:
class Stack:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stack():\n    \"a stack\"\n    s = Stack()\n    assert not s and len(s) == 0\n    for n in [1, 2, 3]:\n        s.push(n)\n    assert repr(s) == \"Stack([1, 2, 3])\" and s.peek() == 3 and s.pop() == 3 and len(s) == 2 and s\n    assert not isinstance(s, list) and not hasattr(s, \"insert\") and not hasattr(s, \"sort\")\n    s.pop(); s.pop()\n    for call in (s.pop, s.peek):\n        try:\n            call()\n            assert False\n        except IndexError as err:\n            assert \"empty stack\" in str(err)")

### Bug hunt: Everybody's basket

Each customer should have their own basket, but adding to Ada's also adds to Bo's, so this prints `['tea', 'rice'] ['tea', 'rice']`. Fix the class so the output is `['tea'] ['rice']`.

In [ ]:
class Customer:
    basket = []

    def __init__(self, name):
        self.name = name

    def add(self, item):
        self.basket.append(item)

ada, bo = Customer("Ada"), Customer("Bo")
ada.add("tea")
bo.add("rice")
print(ada.basket, bo.basket)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a basket each\"\n    expect_output(\"['tea'] ['rice']\")")

### Bug hunt: Lost in a set

`Card` defines `__eq__`, so Python sets its `__hash__` to `None`, and putting cards in a set crashes. Make cards hashable, consistently with equality, so this prints `2`.

In [ ]:
class Card:
    def __init__(self, rank, suit):
        self.rank, self.suit = rank, suit

    def __eq__(self, other):
        if not isinstance(other, Card):
            return NotImplemented
        return (self.rank, self.suit) == (other.rank, other.suit)

hand = {Card("A", "spades"), Card("A", "spades"), Card("7", "hearts")}
print(len(hand))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"hashable cards\"\n    expect_output(\"2\")\n    assert hash(Card(\"K\", \"clubs\")) == hash(Card(\"K\", \"clubs\"))")

### Bug hunt: The missing parent

`Lorry.__init__` never calls the parent's, so `reg` is never set and `describe` crashes. Fix it so this prints `KJA 123: 18 tonnes`.

In [ ]:
class Vehicle:
    def __init__(self, reg):
        self.reg = reg

    def describe(self):
        return self.reg

class Lorry(Vehicle):
    def __init__(self, reg, tonnes):
        self.tonnes = tonnes

    def describe(self):
        return f"{super().describe()}: {self.tonnes} tonnes"

print(Lorry("KJA 123", 18).describe())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"super().__init__\"\n    expect_output(\"KJA 123: 18 tonnes\")")

### Puzzle: Equality done properly

Arrange the lines into a `__eq__` and `__hash__` for a class with `name` and `year`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Box:
    count = 0

    def __init__(self, items=()):
        self.items = list(items)
        Box.count += 1

    def __len__(self):
        return len(self.items)

    def __bool__(self):
        return True

    def __add__(self, other):
        return Box(self.items + other.items)

a = Box([1, 2])
b = Box()
c = a + b + a

p = ___   # len(c)
q = ___   # Box.count
r = ___   # bool(b)
s = ___   # [len(x) for x in (a, b, c)]
t = ___   # c.items is a.items

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4 and q == 4 and r is True, \"p, q and r\"\n    assert s == [2, 0, 4] and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a polynomial

Write `Poly`, a polynomial with whole-number coefficients, made from a list of coefficients lowest power first: `Poly([1, 0, 3])` is 3x² + 1. Trailing zeros are removed (`Poly([0, 0])` has the coefficient list `[]`, the zero polynomial). Give it:

- `coeffs` (the cleaned list), `degree` (a property: the highest power, or `-1` for zero);
- `__call__(x)`, its value at `x`;
- `__add__` and `__mul__` with other polynomials and with ints (an int is a constant polynomial), and `__radd__`/`__rmul__`;
- `__eq__` (and `__hash__`, using a tuple of the coefficients), and `__repr__` like `Poly([1, 0, 3])`;
- `__str__`: terms from the highest power down, like `3x^2 - x + 1`, with these rules: a coefficient of 1 or -1 isn't shown except in the constant term; `x^1` is written `x`; zero terms are left out; a negative term after the first is written ` - ...`; the zero polynomial is `0`.

In [ ]:
class Poly:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"basics\"\n    p = Poly([1, 0, 3, 0, 0])\n    assert p.coeffs == [1, 0, 3] and p.degree == 2 and Poly([0]).degree == -1 and Poly([]).coeffs == []\n    assert p(2) == 13 and p(0) == 1 and Poly([])(5) == 0\n    assert repr(p) == \"Poly([1, 0, 3])\" and Poly([1, 2]) == Poly([1, 2, 0]) and len({Poly([1]), Poly([1, 0])}) == 1\ndef test_maths():\n    \"arithmetic\"\n    a, b = Poly([1, 1]), Poly([-1, 1])\n    assert a * b == Poly([-1, 0, 1]) and a + b == Poly([0, 2]) and a + 4 == Poly([5, 1]) and 4 + a == Poly([5, 1])\n    assert 2 * a == Poly([2, 2]) and a * 0 == Poly([]) and (a + Poly([-1, -1])).degree == -1\ndef test_str():\n    \"printing\"\n    cases = [([1, 0, 3], \"3x^2 + 1\"), ([1, -1, 3], \"3x^2 - x + 1\"), ([0, 1], \"x\"), ([-1], \"-1\"), ([0, -1], \"-x\"), ([], \"0\"),\n             ([5, 0, 0, -2], \"-2x^3 + 5\"), ([0, 0, 1, 1], \"x^3 + x^2\"), ([-4, 2], \"2x - 4\"), ([1, 1], \"x + 1\")]\n    for coeffs, want in cases:\n        assert str(Poly(coeffs)) == want, f\"{coeffs}: got {str(Poly(coeffs))!r}, expected {want!r}\"")

## How to review

The same rules as before, because they're the ones that work:

1. Try every prediction **before** running it, and write your guess down.
2. When a guess is wrong, find exactly why before moving on.
3. Keep a list of everything that surprised you. That list is your revision plan for Friday's exam.

## Classes and instances

A class bundles data (attributes) with the operations on it (methods). The essentials from Weeks 28 and 29:

- `__init__` sets up a new instance; `self` is the instance a method was called on, so `obj.method(x)` is `Class.method(obj, x)`.
- **Instance attributes** (`self.x = ...`) belong to one object; **class attributes** (in the class body) are shared by all of them, and an instance attribute with the same name hides the class one.
- **Invariants** are rules that are always true of an instance between method calls. Keep them by checking before changing, and by marking internal names with a leading underscore.
- A **property** looks like an attribute but runs a method, so a class can compute a value, or check one being set, without changing how callers use it.

`tick` returns `self`, so calls can be chained. `b.made = 99` created an **instance** attribute on `b` that hides the class attribute; `Counter.made` and `a.made` still say 2.

Because `__init__` assigns to `self.celsius`, even a new object's value goes through the check: the invariant can't be broken at creation either.

## Special methods

Special methods ("dunder" methods) let your objects work with Python's own syntax and functions (Weeks 29 and 31):

| You write | Python calls |
|---|---|
| `repr(x)`, and the shell, and lists of `x` | `x.__repr__()` (for developers: ideally looks like code that makes it) |
| `str(x)`, `print(x)`, f-strings | `x.__str__()`, falling back to `__repr__` |
| `x == y` | `x.__eq__(y)`; return `NotImplemented` for types you don't know |
| `hash(x)`, sets, dictionary keys | `x.__hash__()`: equal objects must have equal hashes |
| `x < y`, `sorted` | `x.__lt__(y)`; `functools.total_ordering` fills in the rest from `__eq__` and one of them |
| `x + y`, `x * 3`, `3 * x` | `__add__`, `__mul__`, `__rmul__` |
| `bool(x)`, `if x:` | `__bool__`, falling back to `__len__` |
| `x(...)` | `__call__` |

Three details worth remembering: defining `__eq__` without `__hash__` makes instances **unhashable**; `sum` starts from `0` unless you give it a start value; and `Money(5) == 5` is `False`, not an error, because returning `NotImplemented` lets Python try the other side and then fall back to "not equal".

## Containers

Week 31's container protocols let an object act like a sequence or a collection:

| Behaviour | Method |
|---|---|
| `len(x)` | `__len__` |
| `x[i]`, `x[key]`, slices | `__getitem__` |
| `x[i] = v`, `del x[i]` | `__setitem__`, `__delitem__` |
| `item in x` | `__contains__` (falls back to iterating) |
| `for item in x` | `__iter__` (falls back to `__getitem__` with 0, 1, 2, ...) |

`__len__` and `__getitem__` alone gave this class indexing, negative indexes, slices (passed straight to the list), `in`, `for` and `reversed`. That's duck typing (Week 30): Python asks for the methods, not for a particular parent class.

## Inheritance and composition

Week 30 in brief:

- `class Child(Parent):` inherits everything; the child adds methods and **overrides** others. Attribute lookup follows the **MRO**: instance, class, parents, up to `object`.
- `super().method(...)` runs the next class's version in the MRO on the same `self`. In `__init__`, call `super().__init__(...)` so the parent sets up its part.
- `isinstance` respects inheritance; prefer it to `type(x) == ...`.
- **Composition** ("has a") is often better than inheritance ("is a"): a class that **holds** another object and delegates to it can swap that object, and isn't tied to every detail of a parent.

`super()` in `B` doesn't mean "`B`'s parent": it means "the next class in the MRO of the object", and for a `D` that's `C`. That's how each class in a diamond runs exactly once.

## The classic mistakes

1. **A mutable class attribute used as per-instance data**: `items = []` in the class body is one list shared by every instance. Create it in `__init__`.
2. **Forgetting `super().__init__(...)`**: the parent's attributes are never set, and an `AttributeError` appears later, somewhere else.
3. **`__eq__` without `__hash__`**: instances can't go in sets or be dictionary keys. And never make the hash depend on something that can change while the object is in a set.
4. **`__repr__` that returns something other than a string**, or `__eq__` that raises for other types instead of returning `NotImplemented`.
5. **Inheriting for code reuse when the "is a" relationship isn't true**: a `Stack(list)` inherits `insert` and `sort`, which break the idea of a stack. Compose instead.
6. **Calling `self.method()` instead of `super().method()`** inside an override: endless recursion.

## Practice

## Quiz

## Daily challenge

## Before you go

Look over your list of surprises from today, and for each one write a two-line example that shows the right behaviour. Tomorrow: types and design, from dataclasses and protocols to SOLID and robust programs.